In [1]:
# intall the required libraries. 

!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

   ---------------------------------------- 0.0/4.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.7 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.7 MB ? eta -:--:--
   ---- ----------------------------------- 0.5/4.7 MB 1.2 MB/s eta 0:00:04
   ------ --------------------------------- 0.8/4.7 MB 1.0 MB/s eta 0:00:04
   ------ --------------------------------- 0.8/4.7 MB 1.0 MB/s eta 0:00:04
   -------- ------------------------------- 1.0/4.7 MB 831.8 kB/s eta 0:00:05
   -------- ------------------------------- 1.0/4.7 MB 831.8 kB/s eta 0:00:05
   ----------- ---------------------------- 1.3/4.7 MB 819.9 kB/s eta 0:00:05
   ----------- ---------------------------- 1.3/4.7 MB 819.9 kB/s eta 0:00:05
   ------------- -------------------------- 1.6/4.7 MB 757.2 kB/s eta 0:00:05
   ------------- -------------------------- 1.6/4.7 MB 757.2 kB/s eta 0:00:05
   --------------- ------------------------ 1.8/4.7 MB 764.7 kB/s eta 0:00:04
   --------------- 

In [3]:
!pip install -U ddgs


   ---------------------- ----------------- 4/7 [fake-useragent]
   ---------------------------------- ----- 6/7 [ddgs]
   ---------------------------------------- 7/7 [ddgs]



### DuckDuckGo Search.

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

result = search_tool.invoke('What are the top news in india ?')

print(result)

Times of India brings the Latest News & Breaking News Headlines from India & around the World. Read Latest News Today on Sports, Business, Health & Fitness, Bollywood & Entertainment, Blogs ... Get the latest breaking news from India covering politics, economy, social issues, technology, sports, and more. Stay updated with in-depth analysis and top headlines from across the country. Breaking news from India and the World. Top news stories and videos on Politics, Business, Entertainment, Technology, Sports, Health. Get the latest India news, breaking headlines, political developments, government updates, weather alerts, crime news and state coverage at CNBC TV18. Get the in-depth latest India news today with realtime live updates, breaking headlines, political coverage, photos, videos & top stories only on News24


In [8]:
print(search_tool.args)
print(search_tool.description)
print(search_tool.name)

{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
duckduckgo_search


### Shell tool.

In [9]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

result = shell_tool.invoke('ls')

print(result)

Executing command:
 ls
'ls' is not recognized as an internal or external command,
operable program or batch file.



C:\Users\Mohd Uzaif\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


## Custom Tools in Langchain. 

In [7]:
from langchain_community.tools import tool

### Method - 1  Custom tool using tool decorator.

In [9]:
# Step 1 : Create the function. 
def multiply(a, b): 
    """ Multiply two numbers """
    return a * b         

In [10]:
#Step 2 :  add type hints in the function we define in the first step. 
def multiply(a : int, b : int) -> int:
    """ Multiply two number """
    return a * b   

In [11]:
# Step 3 : Add the decorator of tool on the given function in step 2. 
@tool
def multiply(a : int, b : int) -> int: 
    """ Multiply two number"""
    return a * b

In [12]:
result = multiply.invoke({
    "a" : 3, 
    "b" : 4
})

In [13]:
result

12

In [17]:
# print some info about this function. 
print(multiply.name) 
print(multiply.description)
print(multiply.args)

multiply
Multiply two number
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [19]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two number', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


### Method - 2 Create custom tool using StructuredTool & Pydantic.

In [20]:
from langchain_community.tools import StructuredTool
from pydantic import BaseModel, Field

In [22]:
# create the base model class. 
class MultiplyInput(BaseModel): 
    a : int = Field(description = 'This is the first number')
    b : int = Field(description = 'This is the second number')

In [24]:
# create the custom code in function. 
def multiply_func(a : int, b : int) -> int: 
    return a * b   

In [26]:
multiply_tool = StructuredTool.from_function(
    func = multiply_func,  
    name = "multiply",  
    description = "Multiply two numbers", 
    args_schema = MultiplyInput
)

In [27]:
result = multiply_tool.invoke({
    'a' : 3, 
    'b' : 5
})

In [28]:
print(result)

15


In [29]:
# print the tool related stuff. 
print(multiply_tool.name)        
print(multiply_tool.description)
print(multiply_tool.args)

multiply
Multiply two numbers
{'a': {'description': 'This is the first number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'This is the second number', 'title': 'B', 'type': 'integer'}}


### Method - 3 Using BaseModel class

In [30]:
from langchain.tools import BaseTool
from typing import Type

In [32]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to add")
    b: int = Field(description="The second number to add")

In [33]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [34]:
multiply_tool = MultiplyTool()

In [35]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Toolkit

In [36]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [37]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [38]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers
